# **＜重要＞本章で使用するファイル**

- customer_survey_dirty.csv

これのファイルを左サイドバーのファイルからアップロードしてお使いください。（p.20参照）

Google ドライブにファイルを置き、マウントして使うこともできます（p.21参照）。その場合は次の点に注意してください。

- ファイルのパスは、各自のフォルダに合わせて書き換えてください
- ドライブ上のファイルは各自で管理してください
- ファイルを書き込むコード（to_csv() など）を実行すると、パスを誤ったときに既存のファイルを上書きするおそれがあります

# Menu.32 生データの全体像を把握しよう

全体像を把握する

In [ ]:
import pandas as pd

df = pd.read_csv("customer_survey_dirty.csv")

# データの形状確認
print("列数･行数", df.shape)
print(df.head())

データの構造を確認する

In [ ]:
print(df.info())

# Menu.33 正しいデータ型に変換しよう

不要な文字を取り除く

In [ ]:
import pandas as pd

df = pd.read_csv("customer_survey_dirty.csv")

# 余計な文字を除去
df["income"] = df["income"].str.replace("万円", "") # "万円"を除去
df["satisfaction"] = df["satisfaction"].str.replace("点", "") # "点"を除去
df["purchase_amount"] = df["purchase_amount"].str.replace("¥", "") # "¥"を除去
df["purchase_amount"] = df["purchase_amount"].str.replace(",", "") # ","を除去

# 確認
df_check = df[["income", "satisfaction", "purchase_amount"]]
print(df_check.head())

object型でsum()メソッドを実行する

In [ ]:
# object型のままsumを実行
total_purchase = df["purchase_amount"].sum()
print(total_purchase)

型を変換する

In [ ]:
# 型変換の実行
df["income"] = df["income"].astype(float)
df["satisfaction"] = df["satisfaction"].astype(float)
df["purchase_amount"] = df["purchase_amount"].astype(float)

# 日付型に変換
df["survey_date"] = pd.to_datetime(df["survey_date"])

# 型の確認
print(df.dtypes)

購入金額の合計と平均を確認する

In [ ]:
# 数値計算の実行
sum_purchase = df["purchase_amount"].sum()
avg_purchase = df["purchase_amount"].mean()
print(sum_purchase)
print(avg_purchase)

# Menu.34 表記ゆれを修正しよう

表記ゆれを確認する

In [ ]:
import pandas as pd

df = pd.read_csv("customer_survey_dirty.csv")

# "gender"列のユニークな値を確認
gender_count = df["gender"].value_counts()
print(gender_count)

表記ゆれを修正する

In [ ]:
# "男"を"男性"に置き換え
df["gender_temp"] = df["gender"].replace("男", "男性")

# 置き換え後の値を確認
gender_count = df["gender_temp"].value_counts()
print(gender_count)

表記ゆれを一度で変換する

In [ ]:
# 変換ルールを辞書で定義
gender_mapping = {
    "男": "男性",
    "M": "男性",
    "male": "男性",
    "女": "女性",
    "F": "女性",
    "female": "女性"
}

# 表記ゆれを修正
df["gender_normalized"] = df["gender"].replace(gender_mapping)

# 結果を確認
gender_count = df["gender_normalized"].value_counts()
print(gender_count)

表記ゆれ修正前後を比較する

In [ ]:
print("修正前のデータで集計")
age_by_gender_before = df.groupby("gender")["age"].mean()
print(age_by_gender_before)

print("修正後のデータで集計")
age_by_gender_after = df.groupby("gender_normalized")["age"].mean()
print(age_by_gender_after)

# Menu.35 大文字・小文字統一と空白除去でデータを整えよう

地域データのユニークな値を確認する

In [ ]:
import pandas as pd

df = pd.read_csv("customer_survey_dirty.csv")

# region列のユニークな値の個数を確認
unique_region = df["region"].nunique()
print("地域のユニークな値の個数:", unique_region)

print("実際の地域データ")
region_count = df["region"].value_counts()
print(region_count)

空白を除去してユニークな地域数を確認する

In [ ]:
df["region_stripped"] = df["region"].str.strip()
unique_region_stripped = df["region_stripped"].nunique()
print("空白除去後のユニークな地域数:", unique_region_stripped)

大文字・小文字を統一する

In [ ]:
# 大文字･小文字を小文字に統一 (region列)
df["region_lowercase"] = df["region_stripped"].str.lower()
unique_region_lowercase = df["region_lowercase"].nunique()
print("小文字化後のユニークな地域数:", unique_region_lowercase)

region_count = df["region_lowercase"].value_counts() # 各地域の件数を確認
print(region_count)

## Column メソッドチェーンで複数の処理を連続実行する

小文字化と空白除去をまとめて実行する

In [ ]:
# 小文字化と空白除去を連続で実行
df["region_normalized"] = df["region"].str.lower().str.strip()
unique_region_normalized = df["region_normalized"].nunique()
print(f"正規化後のユニークな地域数: {unique_region_normalized}")

# Menu.36 重複データの問題を発見しよう

重複行を確認する

In [ ]:
import pandas as pd

df = pd.read_csv("customer_survey_dirty.csv")

# 重複している行の数
df_duplicated = df.duplicated()
duplicate_count = df_duplicated.sum()
print("重複行の数", duplicate_count)

## Column duplicated()のkeep引数

keep引数で基準を変える

In [ ]:
df_duplicated_first = df.duplicated(keep="first") # 最初の行以外をTrue（デフォルト）
df_duplicated_last = df.duplicated(keep="last") # 最後の行以外をTrue
df_duplicated_all = df.duplicated(keep=False) # 重複するすべての行をTrue

特定の列を基準に重複を検出する

In [ ]:
# customer_id列が重複している行の数
customer_id_duplicates = df.duplicated(subset=["customer_id"]).sum()
print("customer_idの重複数", customer_id_duplicates)

完全重複行を削除する

In [ ]:
# 完全重複行を削除
df_no_duplicates = df.drop_duplicates()
print("削除後の行数", len(df_no_duplicates))

特定の列を基準に重複を削除する

In [ ]:
# customer_idの重複を削除
df_unique_customers = df.drop_duplicates(subset=["customer_id"])
print("削除後の行数", len(df_unique_customers))

## Column 最後の行を残したい場合（keep="last"）

最後の行を残して重複を削除する

In [ ]:
# customer_idの重複を削除(最後の行を残す)
df_latest_response = df.drop_duplicates(subset=["customer_id"], keep="last")

# Menu.37 外れ値を検出しよう

基本統計量を確認する

In [ ]:
import pandas as pd

df = pd.read_csv("customer_survey_dirty.csv")

# 文字列の除去と型変換
df["income"] = df["income"].str.replace("万円", "").astype(float)

# 重複削除
df = df.drop_duplicates()

# 基本統計量による確認
print(df.describe())

現実的ではない外れ値を検出する

In [ ]:
# 現実的でない年齢(0歳未満 or 150歳超)
age_0_150 = (df["age"] < 0) | (df["age"] > 150)
unrealistic_age = df[age_0_150]
print("0歳未満 or 150歳超:", len(unrealistic_age))

# 現実的でない年収(マイナスの値)
income_negative = df["income"] < 0
unrealistic_income = df[income_negative]
print("年収(マイナス):", len(unrealistic_income))

外れ値の境界を計算する

In [ ]:
income_data = df["income"]

# Q1, Q3, IQRを計算
Q1 = income_data.quantile(0.25)
Q3 = income_data.quantile(0.75)
IQR = Q3 - Q1

print("Q1 (25%)", Q1)
print("Q3 (75%)", Q3)
print("IQR", IQR)
# 外れ値の境界を計算
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print("下限", lower_bound)
print("上限", upper_bound)

統計的に外れ値を検出する

In [ ]:
# 外れ値を検出
outliers = income_data[ (income_data < lower_bound) |(income_data > upper_bound)]

print("検出された外れ値の数:", len(outliers))
print("外れ値の具体例")
print(outliers.sort_values(ascending=False).head(3))

## Column pandasの裏側で動くNumPy

NumPyで四分位数を計算する

In [ ]:
import numpy as np

# NumPyで四分位数を計算(欠損値を除外して渡す)
Q1_np = np.percentile(income_data.dropna(), 25)
Q3_np = np.percentile(income_data.dropna(), 75)
print("Q1:", Q1_np, "Q3:", Q3_np)

NumPy配列を取り出す

In [ ]:
# pandasのSeriesからNumPy配列を取り出す
print(income_data.values[:5])
print(type(income_data.values))

# Menu.38 外れ値を修正・削除しよう

外れ値の境界を計算する

In [ ]:
import pandas as pd
df = pd.read_csv("customer_survey_dirty.csv")

# 前処理
df["income"] = df["income"].str.replace("万円", "")
df["income"] = df["income"].astype(float)

# 前のMenu37で計算したIQR境界を再計算
income_data = df["income"]
Q1 = income_data.quantile(0.25)
Q3 = income_data.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("下限", lower_bound)
print("上限", upper_bound)

外れ値を削除する

In [ ]:
# IQR法の境界外のデータを削除
# incomeがNaNの行は保持する(外れ値ではないため、別途欠損値処理で扱う)
df_no_outliers = df[(df["income"] >= lower_bound) & (df["income"] <= upper_bound) | df["income"].isna()]
print("削除後の行数:", len(df_no_outliers))

外れ値削除前後の統計量を確認する

In [ ]:
# 削除前後の基本統計量の比較
print("削除前")
print(df["income"].describe())

print("削除後")
print(df_no_outliers["income"].describe())

外れ値を修正する

In [ ]:
# 戦略2:外れ値の修正(クリッピング)
df_clipped = df.copy()

# 外れ値を境界値に修正
df_clipped["income"] = df_clipped["income"].clip( lower=lower_bound, upper=upper_bound)

# 修正前後の基本統計量の比較
print("修正前")
print(df["income"].describe())

print("修正後")
print(df_clipped["income"].describe())

# Menu.39 欠損値を処理しよう

欠損値を数える

In [ ]:
import pandas as pd

df = pd.read_csv("customer_survey_dirty.csv")

# 余計な文字を除去
df["income"] = df["income"].str.replace("万円", "") # "万円"を除去
df["satisfaction"] = df["satisfaction"].str.replace("点", "") # "点"を除去
df["purchase_amount"] = df["purchase_amount"].str.replace("¥", "") # "¥"を除去
df["purchase_amount"] = df["purchase_amount"].str.replace(",", "") # ","を除去

# 型変換
df["income"] = df["income"].astype(float)
df["satisfaction"] = df["satisfaction"].astype(float)
df["purchase_amount"] = df["purchase_amount"].astype(float)

# 各列の欠損値の数
missing_counts = df.isnull().sum()
print(missing_counts)

欠損値を含む行を削除する

In [ ]:
# 欠損値を含む行をすべて削除
df_dropped = df.dropna()
print("削除後のデータ件数", len(df_dropped))

特定の列の欠損値を含む行を削除する

In [ ]:
# 特定の列に欠損値がある行を削除
important_columns = ["age", "satisfaction"]
df_partial_dropped = df.dropna(subset=important_columns)
print("削除後のデータ件数", len(df_partial_dropped))

欠損値を置き換える

In [ ]:
# 平均値で補完
mean_val = df["age"].mean()
df_mean = df["age"].fillna(mean_val)

# 中央値で補完
median_val = df["age"].median()
df_median = df["age"].fillna(median_val)

# 固定値(0)で補完
df_zero = df["age"].fillna(0)

欠損値を複数回補完する

In [ ]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# 数値列のみを抽出
numeric_cols = ["age", "income", "satisfaction", "purchase_amount"]
df_numeric = df[numeric_cols].copy()

# 補完済みデータセットを格納するリスト
imputed_datasets = []

# 補完済みデータセットを5個作成
m = 5
for i in range(m):
  imputer = IterativeImputer(
          max_iter=10,
          random_state=i,
          sample_posterior=True
  )
  imputed_data = imputer.fit_transform(df_numeric)
  df_imputed = pd.DataFrame(
      imputed_data,
      columns=numeric_cols
  )
  imputed_datasets.append(df_imputed)

補完済みデータセットごとの標準偏差を確認する

In [ ]:
print("標準偏差の比較")
print("元データ:", round(df["age"].std(), 2))
print("平均値補完:", round(df_mean.std(), 2))
for i, imputed_df in enumerate(imputed_datasets, start=1):
    print("補完データ" + str(i) + ":", round(imputed_df["age"].std(), 2))

補完済みデータセットごとに平均値を求める

In [ ]:
age_means = []
for imputed_df in imputed_datasets:
    age_mean = imputed_df["age"].mean()
    age_means.append(float(round(age_mean, 2)))
print(age_means)

分析結果を統合する

In [ ]:
pooled_age_mean = np.mean(age_means)
print("統合後のage平均値:", round(pooled_age_mean, 2))